# Spam Text - Megoldás

In [ ]:
# @title Képek és a Modell file letöltése

!gdown -qq 1-MtzrY1JqJEfvxuSASobGOYWAoZ3NeGu
!gdown -qq 1I7IJZ-ieEXCp-AVmrUzaFXPjLCNNLvKc

In [ ]:
import torch
import re
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

In [ ]:
df = pd.read_csv('./sms.csv')
print("Beérkezett SMS-ek száma", len(df))
df.head()

Beérkezett SMS-ek száma 5572


,category,text
0,legit,"Go until jurong point, crazy.. Available only ..."
1,legit,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,legit,U dun say so early hor... U c already then say...
4,legit,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
def clean_text(text):
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.lower()

df['clean_reviews'] = df['text'].apply(clean_text)
label_encoder = LabelEncoder()
df['labels'] = label_encoder.fit_transform(df['category'])

texts = df['clean_reviews']
labels = df['labels']

In [ ]:
df.head()

,category,text,clean_reviews,labels
0,legit,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,0
1,legit,Ok lar... Joking wif u oni...,ok lar joking wif u oni,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,1
3,legit,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,0
4,legit,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...,0


In [ ]:
def build_vocab(texts):
    counter = Counter()
    for text in texts:
        counter.update(text.split())
    return {word: i + 1 for i, word in enumerate(counter)}

vocab = build_vocab(texts)

print(len(vocab))

def tokenize(text, vocab):
    return [vocab[word] for word in text.split() if word in vocab]

train_sequences = [torch.tensor(tokenize(text, vocab)) for text in texts]

train_sequences_padded = pad_sequence(train_sequences, batch_first=True)


9476


In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

train_dataset = ReviewDataset(train_sequences_padded, torch.tensor(np.array(labels)))

In [ ]:
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, output_dim):
        super(SentimentModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.dropout = nn.Dropout(0.2)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(embedding_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = self.global_pool(x).squeeze(2)
        x = self.dropout(x)
        return self.fc(x)

model = SentimentModel(len(vocab) + 1, 16, len(label_encoder.classes_))

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

for epoch in range(100):
    model.train()
    for texts, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')


Epoch 1, Loss: 0.5272536277770996
Epoch 2, Loss: 0.8063808679580688
Epoch 3, Loss: 0.14872147142887115
Epoch 4, Loss: 0.1405770182609558
Epoch 5, Loss: 0.15195909142494202
Epoch 6, Loss: 0.17352527379989624
Epoch 7, Loss: 1.1596328020095825
Epoch 8, Loss: 0.09701723605394363
Epoch 9, Loss: 0.4410915970802307
Epoch 10, Loss: 0.48622289299964905
Epoch 11, Loss: 0.6595517992973328
Epoch 12, Loss: 0.155732199549675
Epoch 13, Loss: 0.3393194377422333
Epoch 14, Loss: 0.1398908793926239
Epoch 15, Loss: 0.20770946145057678
Epoch 16, Loss: 0.42260727286338806
Epoch 17, Loss: 0.05836627632379532
Epoch 18, Loss: 0.05268731713294983
Epoch 19, Loss: 0.17046427726745605
Epoch 20, Loss: 0.1910018026828766
Epoch 21, Loss: 0.04731999710202217
Epoch 22, Loss: 0.07623262703418732
Epoch 23, Loss: 0.08400619029998779
Epoch 24, Loss: 0.02169676311314106
Epoch 25, Loss: 0.015201841481029987
Epoch 26, Loss: 0.016826612874865532
Epoch 27, Loss: 0.010472574271261692
Epoch 28, Loss: 0.03638194501399994
Epoch 29,

In [ ]:
torch.save(model.state_dict(), 'jános_email_modell.pth')
model.load_state_dict(torch.load('jános_email_modell.pth'))

In [ ]:
def predict(text, model, vocab):
    model.eval()
    tokens = torch.tensor([vocab.get(t, 0) for t in text.split()])
    prediction = model(tokens.unsqueeze(0))
    return label_encoder.inverse_transform([torch.argmax(prediction)])

review_1 = "Hey, János. I have a crazy hiking opportunity for you in the best place on Earth, the Himalayas. I can be your hiking partner right away, just send me your bank information and personal details, and let's get started."
review_2 = "Nah I don't think he goes to usf, he lives aro."
print(predict(review_1, model, vocab))
print(predict(review_2, model, vocab))